# RAMF graph-decode diagnostic (Approach A)

Separates two things the training-log comparison cannot:
1. **decoder ceiling** — graph-match AAR on the *ground-truth* `clean` atom14.
2. **actual** — graph-match AAR on the model *prediction* `pre`.
3. **where it breaks** — predicted virtual-atom distance to N/O (marker_eps),
   and predicted real-atom RMSD vs clean (fit quality).

VERIFIED fact about the dumped pt: `clean` carries markers ONLY on CDR residues;
non-CDR residues are standard atom14 with unoccupied slots = sentinel (~106). So
CDR = residues whose reconstructed n_real < 14. The alignment self-check runs on
those residues only.

In [1]:
import os, sys, glob
import numpy as np
import torch

ROOT = '/root/private_data/luog/codex/IgGM2'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from IgGM.protein.prot_constants import ATOM_NAMES_PER_RESD, RESD_MAP_1TO3, RESD_NAMES_1C
from src.iggm_lightning.atom14_sync import Atom14SeqSync

SEQ = 'QVQLQESGPGLVRPSQTLSLTCTVSGFHQTGYGVNWVRQPPGRGLEWIGMIWGDGNTDYNSALKSRVTMLKDTSKNQFSLRLSSVTAADTAVYYCAREKDYRLDYWGQGSLVTVSSDIQMTQSPSSLSASVGDRVTITCRWSGPIFNYLAWYQQKPGKAPKLLIYYTTTLADGVPSRFSGSGSGTDYTFTISSLQPEDIATYYCQHFWSTPRTFGQGTKVEIKRKVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL'
L = len(SEQ)
sync = Atom14SeqSync()
n_real_of = {aa: len(ATOM_NAMES_PER_RESD[RESD_MAP_1TO3[aa]]) for aa in RESD_NAMES_1C}

# # Use a SPECIFIC file, or glob all length-L samples. Set FILES as you like.
# FILES = sorted(glob.glob(os.path.join(ROOT, 'see/seefile/*.pt')))
# samples = []
# for f in FILES:
#     d = torch.load(f, map_location='cpu')
#     if tuple(d['pre'].shape) == (1, L, 14, 3):
#         samples.append((f, d))
# print('matched length-%d samples: %d' % (L, len(samples)))
# assert samples, 'no length-matching samples found'

[2026-07-21 11:43:14,129] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


## Cell 1 — CDR inference + alignment self-check (on CDR residues only)

In [2]:
def reconstruct_n_real(c, dup_eps=0.3):
    """[14,3] -> #real atoms; a non-{0,3} slot within dup_eps of N/O is a marker."""
    n_pos, o_pos = c[0], c[3]
    nr = 0
    for s in range(14):
        if s in (0, 3):
            nr += 1; continue
        if min(float((c[s]-n_pos).norm()), float((c[s]-o_pos).norm())) > dup_eps:
            nr += 1
    return nr

def marker_residue_mask(clean, dup_eps=0.3):
    return torch.tensor([reconstruct_n_real(clean[i], dup_eps) < 14 for i in range(clean.shape[0])])

f0 = os.path.join(ROOT, 'see/seefile/S0720_1784521052.pt')
d0 = torch.load(f0, map_location='cpu')

# f0, d0 = samples[0]
clean0 = d0['clean'][0]
cdr0 = marker_residue_mask(clean0)
theo = torch.tensor([n_real_of.get(c, 14) for c in SEQ])
recon = torch.tensor([reconstruct_n_real(clean0[i]) for i in range(L)])
idx = torch.where(cdr0)[0]
print('inferred CDR (marker-bearing) residues:', idx.tolist())
print('their AA in SEQ                        :', ''.join(SEQ[i] for i in idx.tolist()))
agree = float((theo[cdr0] == recon[cdr0]).float().mean())
print('n_real agreement ON CDR residues: %.3f (%d/%d)' % (agree, int((theo[cdr0]==recon[cdr0]).sum()), int(cdr0.sum())))
assert agree > 0.7, 'ALIGNMENT FAILED on CDR residues; order/crop wrong -> results void.'
print('alignment OK (CDR region + sequence order consistent)')

/tmp/ipykernel_26507/1435834455.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d0 = torch.load(f0, map_location='cpu')


inferred CDR (marker-bearing) residues: [25, 26, 27, 28, 29, 30, 31, 52, 53, 54, 55, 97, 98, 99, 100, 101, 102, 103, 104, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 165, 166, 167, 168, 169, 170, 171, 204, 205, 206, 208, 209, 210, 211, 212]
their AA in SEQ                        : GFHQTGYGDGNEKDYRLDYRWSGPIFNYLAYTTTLADQHFSTPRT
n_real agreement ON CDR residues: 0.867 (39/45)
alignment OK (CDR region + sequence order consistent)


## Cell 2 — where it breaks: marker distance + real-atom RMSD

In [3]:
vdist, rmsd_all = [], []

f0 = os.path.join(ROOT, 'see/seefile/S0720_1784521052.pt')
d0 = torch.load(f0, map_location='cpu')

pre, clean = d0['pre'][0], d0['clean'][0]
cdr = marker_residue_mask(clean)
for i in torch.where(cdr)[0].tolist():
    nr = reconstruct_n_real(clean[i])
    for s in range(nr, 14):
        on_n = (clean[i, s]-clean[i, 0]).norm() <= (clean[i, s]-clean[i, 3]).norm()
        anc = pre[i, 0] if on_n else pre[i, 3]
        vdist.append(float((pre[i, s]-anc).norm()))
    if nr > 0:
        rmsd_all.append(float(((pre[i,:nr]-clean[i,:nr])**2).sum(-1).mean().sqrt()))

vdist = np.array(vdist); rmsd_all = np.array(rmsd_all)
print('predicted VIRTUAL-atom dist to target N/O (A):')
print('  mean %.2f  median %.2f  p90 %.2f  max %.2f' % (vdist.mean(), np.median(vdist), np.percentile(vdist,90), vdist.max()))
for e in (0.5, 1.0, 1.5):
    print('  frac removable at marker_eps=%.1f : %.3f' % (e, (vdist<e).mean()))
print('predicted REAL-atom RMSD vs clean (A):')
print('  mean %.2f  median %.2f  p90 %.2f' % (rmsd_all.mean(), np.median(rmsd_all), np.percentile(rmsd_all,90)))

# vdist, rmsd_all = [], []
# for f, d in samples[:20]:
#     pre, clean = d['pre'][0], d['clean'][0]
#     cdr = marker_residue_mask(clean)
#     for i in torch.where(cdr)[0].tolist():
#         nr = reconstruct_n_real(clean[i])
#         for s in range(nr, 14):
#             on_n = (clean[i, s]-clean[i, 0]).norm() <= (clean[i, s]-clean[i, 3]).norm()
#             anc = pre[i, 0] if on_n else pre[i, 3]
#             vdist.append(float((pre[i, s]-anc).norm()))
#         if nr > 0:
#             rmsd_all.append(float(((pre[i,:nr]-clean[i,:nr])**2).sum(-1).mean().sqrt()))

# vdist = np.array(vdist); rmsd_all = np.array(rmsd_all)
# print('predicted VIRTUAL-atom dist to target N/O (A):')
# print('  mean %.2f  median %.2f  p90 %.2f  max %.2f' % (vdist.mean(), np.median(vdist), np.percentile(vdist,90), vdist.max()))
# for e in (0.5, 1.0, 1.5):
#     print('  frac removable at marker_eps=%.1f : %.3f' % (e, (vdist<e).mean()))
# print('predicted REAL-atom RMSD vs clean (A):')
# print('  mean %.2f  median %.2f  p90 %.2f' % (rmsd_all.mean(), np.median(rmsd_all), np.percentile(rmsd_all,90)))

/tmp/ipykernel_26507/2656639508.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d0 = torch.load(f0, map_location='cpu')


predicted VIRTUAL-atom dist to target N/O (A):
  mean 0.96  median 0.86  p90 1.71  max 2.62
  frac removable at marker_eps=0.5 : 0.194
  frac removable at marker_eps=1.0 : 0.593
  frac removable at marker_eps=1.5 : 0.832
predicted REAL-atom RMSD vs clean (A):
  mean 13.14  median 12.73  p90 16.09


## Cell 3 — AAR: clean (ceiling) vs pre (actual) vs old N/O count, sweeping marker_eps

In [4]:
def aar(decoded, mask):
    hit = tot = 0
    for i in range(L):
        if not bool(mask[i]):
            continue
        tot += 1; hit += int(decoded[i] == SEQ[i])
    return hit, tot

for eps in (0.5, 1.0, 1.5):
    hc = hp = hn = t = 0

    
    pre, clean = d0['pre'][0], d0['clean'][0]
    cmask = marker_residue_mask(clean)
    cmsk14 = torch.ones(L, 14)
    dec_clean = sync.decode_cdr_sequence_graph(SEQ, clean, cmsk14, cmask, descriptor='fused', gw_eps=0.5, gw_iters=20, topk=5, marker_eps=eps)
    dec_pre   = sync.decode_cdr_sequence_graph(SEQ, pre,   cmsk14, cmask, descriptor='fused', gw_eps=0.5, gw_iters=20, topk=5, marker_eps=eps)
    dec_cnt   = sync.decode_cdr_sequence(SEQ, pre, cmsk14, cmask)
    a,tt = aar(dec_clean, cmask); hc += a
    b,_  = aar(dec_pre,   cmask); hp += b
    c,_  = aar(dec_cnt,   cmask); hn += c
    t += tt
    print('marker_eps=%.1f:  clean_AAR=%.3f  pre_AAR=%.3f  oldNO_AAR=%.3f  (n_res=%d)' % (eps, hc/t, hp/t, hn/t, t))

# def aar(decoded, mask):
#     hit = tot = 0
#     for i in range(L):
#         if not bool(mask[i]):
#             continue
#         tot += 1; hit += int(decoded[i] == SEQ[i])
#     return hit, tot

# N_EVAL = 10  # graph decode is slow; raise for a tighter estimate
# for eps in (0.5, 1.0, 1.5):
#     hc = hp = hn = t = 0
#     for f, d in samples[:N_EVAL]:
#         pre, clean = d['pre'][0], d['clean'][0]
#         cmask = marker_residue_mask(clean)
#         cmsk14 = torch.ones(L, 14)
#         dec_clean = sync.decode_cdr_sequence_graph(SEQ, clean, cmsk14, cmask, descriptor='fused', gw_eps=0.5, gw_iters=20, topk=5, marker_eps=eps)
#         dec_pre   = sync.decode_cdr_sequence_graph(SEQ, pre,   cmsk14, cmask, descriptor='fused', gw_eps=0.5, gw_iters=20, topk=5, marker_eps=eps)
#         dec_cnt   = sync.decode_cdr_sequence(SEQ, pre, cmsk14, cmask)
#         a,tt = aar(dec_clean, cmask); hc += a
#         b,_  = aar(dec_pre,   cmask); hp += b
#         c,_  = aar(dec_cnt,   cmask); hn += c
#         t += tt
#     print('marker_eps=%.1f:  clean_AAR=%.3f  pre_AAR=%.3f  oldNO_AAR=%.3f  (n_res=%d)' % (eps, hc/t, hp/t, hn/t, t))

marker_eps=0.5:  clean_AAR=0.444  pre_AAR=0.089  oldNO_AAR=0.333  (n_res=45)
marker_eps=1.0:  clean_AAR=0.444  pre_AAR=0.133  oldNO_AAR=0.333  (n_res=45)
marker_eps=1.5:  clean_AAR=0.178  pre_AAR=0.133  oldNO_AAR=0.333  (n_res=45)


## Cell 4 — how to read it

| clean_AAR | pre_AAR | conclusion |
|---|---|---|
| ~0.9 | ~0.9 | graph decode works; gap is elsewhere (mask/超参) — fixable now |
| ~0.9 | low | **decoder valid, bottleneck = fit precision** -> clean-atom retrain to cash in |
| low | low | decoder/region broken, or CDR conformational diversity makes GW un-separable -> keep N/O |

Cross-check with Cell 2:
* virtual-atom p90 >> chosen marker_eps -> markers pollute the matrix; raising eps should lift pre_AAR.
* real-atom RMSD mean ~1A+ -> oracle curve (0.3A->0.77, 0.5A->0.56) already explains low pre_AAR -> retrain, not decoder.

The single decisive number is **clean_AAR**: it is the decoder's ceiling on perfect
geometry. High clean_AAR + low pre_AAR overturns "graph decode is useless" — the
problem is that this marker-trained checkpoint cannot emit clean geometry.

## Cell 5 — decisive sanity: does the decoder work on IDEAL templates?

Feeds each CDR residue its own **ideal single-conformer template** (optionally +
Gaussian noise) as the query, through the SAME `decode_cdr_sequence_graph`.

* ideal_AAR ~0.9 while clean_AAR (Cell 3) ~random  =>  decoder is fine; the killer
  is real rotamer diversity vs single-conformer templates (method limit, not a bug).
* ideal_AAR also low  =>  bug in this decode path; fix before any conclusion.

The noise sweep reproduces the oracle curve, so you can map real-atom geometry
error onto expected AAR.

In [5]:
from IgGM.protein.prot_constants import IDEAL_ATOM14_COORDS, IDEAL_ATOM14_MASK

IDEAL = torch.as_tensor(IDEAL_ATOM14_COORDS, dtype=torch.float32)  # [20,14,3]
IMASK = torch.as_tensor(IDEAL_ATOM14_MASK, dtype=torch.float32)    # [20,14]
aa2idx = {aa: i for i, aa in enumerate(RESD_NAMES_1C)}

# Build a synthetic protein whose CDR residues ARE their own ideal templates.
# Non-real slots are filled with markers on N/O exactly like build_supervision,
# so the decoder's marker removal sees the same layout it expects.
def build_ideal_query(seq, cdr_mask, noise=0.0):
    Lq = len(seq)
    q = torch.zeros(Lq, 14, 3)
    for i in range(Lq):
        aa = seq[i]
        k = aa2idx.get(aa)
        if k is None:
            continue
        coords = IDEAL[k].clone()
        nr = int(IMASK[k].sum())
        # markers: non-real slots -> N (slot 0), matches a removable layout
        for s in range(nr, 14):
            coords[s] = coords[0]
        if noise > 0:
            coords[:nr] = coords[:nr] + noise * torch.randn(nr, 3)
        q[i] = coords
    return q

cmask = marker_residue_mask(d0['clean'][0])
cmsk14 = torch.ones(L, 14)

print('IDEAL-template query through the SAME decoder (marker_eps=0.5):')
for noise in (0.0, 0.3, 0.5, 1.0):
    q = build_ideal_query(SEQ, cmask, noise=noise)
    dec = sync.decode_cdr_sequence_graph(SEQ, q, cmsk14, cmask, descriptor='fused',
                                         gw_eps=0.5, gw_iters=20, topk=5, marker_eps=0.5)
    h, t = aar(dec, cmask)
    print('  noise=%.1fA :  ideal_AAR=%.3f  (%d/%d)' % (noise, h/t, h, t))
print('\nreference: clean_AAR (real true geometry) from Cell 3 was ~0.09')
print('if ideal noise=0 >> clean -> decoder OK, real rotamer diversity is the killer.')

IDEAL-template query through the SAME decoder (marker_eps=0.5):
  noise=0.0A :  ideal_AAR=0.889  (40/45)
  noise=0.3A :  ideal_AAR=0.467  (21/45)
  noise=0.5A :  ideal_AAR=0.244  (11/45)
  noise=1.0A :  ideal_AAR=0.111  (5/45)

reference: clean_AAR (real true geometry) from Cell 3 was ~0.09
if ideal noise=0 >> clean -> decoder OK, real rotamer diversity is the killer.
